# Football Market Value — Pipeline Data Engineering
## Notebook de démarrage complet

**Ce notebook lance tout le projet de A à Z :**
1. Démarre l'infrastructure Docker
2. Upload les données en Bronze (MinIO)
3. Déclenche le pipeline Airflow
4. Vérifie les résultats dans PostgreSQL
5. Affiche les insights métier

---
### Prérequis avant de lancer ce notebook
- Docker Desktop installé et **lancé**
- Les données Kaggle téléchargées dans `data/raw/`
- Python 3.8+ avec les librairies installées (`pip install -r requirements.txt`)

```
data/raw/
  Transfermarkt/
    players.csv
    player_valuations.csv
    appearances.csv
    clubs.csv
    competitions.csv
  FIFA 23 Players/
    male_players.csv
```

---
## ÉTAPE 0 — Chargement des librairies

In [ ]:
import subprocess
import time
import boto3
import psycopg2
import pandas as pd
import requests
from botocore.client import Config
import warnings
warnings.filterwarnings('ignore')

print('Librairies chargees avec succes !')


---
## ÉTAPE 1 — Démarrage de l'infrastructure Docker

In [ ]:
print('Lancement de docker-compose...')
result = subprocess.run(
    ['docker-compose', 'up', '-d'],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode == 0:
    print('Docker-compose lance avec succes !')
else:
    print('ERREUR :', result.stderr)

In [ ]:
print('Verification des services Docker...')
result = subprocess.run(
    ['docker-compose', 'ps'],
    capture_output=True, text=True
)
print(result.stdout)

In [ ]:
print('Attente du demarrage des services...')
print('(PostgreSQL, MinIO, Airflow peuvent prendre 2-3 minutes)')
print()

AIRFLOW_URL = 'http://localhost:8081'

# Attente minimale de 30s pour que les conteneurs soient up
time.sleep(30)
print('30s ecoules — verification Airflow en cours...')

# Verification dynamique : toutes les 10s, max 4 minutes
for attempt in range(24):
    try:
        resp = requests.get(f'{AIRFLOW_URL}/health', timeout=5)
        if resp.status_code == 200:
            scheduler = resp.json().get('scheduler', {}).get('status', '?')
            print(f'Airflow pret ! (scheduler: {scheduler})')
            break
    except Exception:
        pass
    print(f'  [{(attempt+1)*10 + 30}s] Airflow pas encore pret, attente...')
    time.sleep(10)
else:
    print("ATTENTION : Airflow n'a pas repondu en 4min.")
    print("Verifier : docker-compose logs airflow-init")

---
## ÉTAPE 2 — Upload des données vers Bronze (MinIO)

In [ ]:
print('Connexion a MinIO...')

s3 = boto3.client(
    's3',
    endpoint_url='http://localhost:9002',
    aws_access_key_id='minioadmin',
    aws_secret_access_key='minioadmin',
    config=Config(signature_version='s3v4'),
    region_name='us-east-1'
)

# Attente que MinIO soit pret (max 60s)
for attempt in range(6):
    try:
        s3.list_buckets()
        print('MinIO pret !')
        break
    except Exception:
        print(f'  MinIO pas encore pret ({attempt+1}/6), attente...')
        time.sleep(10)
else:
    raise Exception("MinIO inaccessible apres 60s. Verifier docker-compose.")

# Creer les buckets si manquants
for bucket in ['bronze', 'silver']:
    existing = [b['Name'] for b in s3.list_buckets()['Buckets']]
    if bucket not in existing:
        s3.create_bucket(Bucket=bucket)
        print(f'Bucket {bucket} cree !')
    else:
        print(f'Bucket {bucket} existe deja OK')

In [ ]:
from pathlib import Path

print('Upload des fichiers CSV vers Bronze...')

raw_path = Path('data/raw')
csv_files = list(raw_path.rglob('*.csv'))

print(f'{len(csv_files)} fichiers CSV trouves')

for file_path in csv_files:
    relative = file_path.relative_to(raw_path)
    s3_key = str(relative).replace('\\', '/')

    try:
        s3.head_object(Bucket='bronze', Key=s3_key)
        print(f'  EXISTE DEJA : {s3_key}')
    except Exception:
        s3.upload_file(str(file_path), 'bronze', s3_key)
        print(f'  UPLOADE : {s3_key}')

print('Upload termine !')

In [ ]:
print('Verification des fichiers dans Bronze...')

response = s3.list_objects_v2(Bucket='bronze')
fichiers = response.get('Contents', [])

print(f'Nombre de fichiers dans Bronze : {len(fichiers)}')
print()
for f in fichiers:
    taille_mb = f['Size'] / (1024 * 1024)
    print(f"  {f['Key']} ({taille_mb:.1f} MB)")

---
## ÉTAPE 3 — Déclenchement du pipeline Airflow

In [ ]:
print('Declenchement du DAG Airflow...')

AUTH = ('admin', 'admin')
DAG_ID = 'football_market_value_pipeline'

# Activer le DAG (au cas ou il soit en pause)
resp_patch = requests.patch(
    f'{AIRFLOW_URL}/api/v1/dags/{DAG_ID}',
    json={'is_paused': False},
    auth=AUTH
)
print(f'DAG active : HTTP {resp_patch.status_code}')

# Declencher le DAG
resp_run = requests.post(
    f'{AIRFLOW_URL}/api/v1/dags/{DAG_ID}/dagRuns',
    json={},
    auth=AUTH
)
print(f'DAG declenche : HTTP {resp_run.status_code}')

if resp_run.status_code == 200:
    run_id = resp_run.json().get('dag_run_id')
    print(f'Run ID : {run_id}')
    print(f'Suivi en direct : {AIRFLOW_URL}')
else:
    print('ERREUR :', resp_run.text)
    print("Verifier que Airflow est bien demarre.")

In [ ]:
print('Attente de la fin du pipeline...')
print('Le pipeline prend environ 10-20 minutes.')
print()
print('Suivi en temps reel :')

max_wait = 80  # 80 x 30s = 40 minutes max
for i in range(max_wait):
    time.sleep(30)

    response = requests.get(
        f'{AIRFLOW_URL}/api/v1/dags/{DAG_ID}/dagRuns?limit=1&order_by=-start_date',
        auth=AUTH
    )

    if response.status_code == 200:
        runs = response.json().get('dag_runs', [])
        if runs:
            state = runs[0].get('state')
            print(f'  [{(i+1)*30}s] Status : {state}')

            if state == 'success':
                print()
                print('PIPELINE TERMINE AVEC SUCCES !')
                break
            elif state == 'failed':
                print()
                print('PIPELINE ECHOUE — Voir Airflow UI pour les logs :')
                print(f'  {AIRFLOW_URL}')
                break

---
## ÉTAPE 4 — Vérification Silver (MinIO Parquet)

In [ ]:
print('Verification de la zone Silver...')
print()

response = s3.list_objects_v2(Bucket='silver', Delimiter='/')
prefixes = response.get('CommonPrefixes', [])

print('Tables Parquet dans Silver :')
for p in prefixes:
    prefix = p['Prefix']
    obj_response = s3.list_objects_v2(Bucket='silver', Prefix=prefix)
    nb = len(obj_response.get('Contents', []))
    print(f'  {prefix} ({nb} fichiers)')

print()
print('Verification du partitionnement par annee :')
response_part = s3.list_objects_v2(
    Bucket='silver',
    Prefix='player_valuations/',
    Delimiter='/'
)
for p in response_part.get('CommonPrefixes', []):
    print(f"  {p['Prefix']}")

---
## ÉTAPE 5 — Vérification Gold (PostgreSQL)

In [ ]:
print('Connexion a PostgreSQL...')

conn = psycopg2.connect(
    host='localhost',
    port=5432,
    database='football',
    user='airflow',
    password='airflow'
)

print('Connecte !')
print()

tables = [
    'dim_player', 'dim_club', 'dim_competition',
    'fact_player_value', 'agg_value_by_position',
    'agg_value_by_league', 'agg_value_by_age',
    'agg_top_players', 'agg_value_by_nationality'
]

print(f'{"Table":<35} {"Lignes":>10}')
print('-' * 47)

cursor = conn.cursor()
total = 0
for table in tables:
    cursor.execute(f'SELECT COUNT(*) FROM {table}')
    count = cursor.fetchone()[0]
    total += count
    print(f'{table:<35} {count:>10,}')

print('-' * 47)
print(f'{"TOTAL":<35} {total:>10,}')

---
## ÉTAPE 6 — Résultats métier

In [ ]:
print('TOP 10 JOUEURS PAR VALEUR MARCHANDE')
print('=' * 60)

df_top = pd.read_sql("""
    SELECT player_name, position, nationality,
           latest_market_value, competition_name
    FROM fact_player_value
    ORDER BY latest_market_value DESC
    LIMIT 10
""", conn)

df_top['latest_market_value'] = df_top['latest_market_value'].apply(
    lambda x: f"{x:,.0f} EUR"
)
print(df_top.to_string(index=False))

In [ ]:
print('VALEUR MARCHANDE MOYENNE PAR POSTE')
print('=' * 50)

df_pos = pd.read_sql("""
    SELECT position,
           ROUND(avg_market_value) AS valeur_moyenne,
           player_count
    FROM agg_value_by_position
    WHERE position != 'Missing'
    ORDER BY avg_market_value DESC
""", conn)

df_pos['valeur_moyenne'] = df_pos['valeur_moyenne'].apply(
    lambda x: f"{x:,.0f} EUR"
)
print(df_pos.to_string(index=False))
print()
print("Insight : Les attaquants sont les plus valorises !")

In [ ]:
print('PIC DE VALEUR PAR AGE — TOP 5 AGES')
print('=' * 50)

df_age = pd.read_sql("""
    SELECT age,
           ROUND(avg_market_value) AS valeur_moyenne,
           player_count
    FROM agg_value_by_age
    ORDER BY avg_market_value DESC
    LIMIT 5
""", conn)

df_age['valeur_moyenne'] = df_age['valeur_moyenne'].apply(
    lambda x: f"{x:,.0f} EUR"
)
print(df_age.to_string(index=False))
print()
print("Insight : Le pic de valeur est entre 24 et 26 ans !")

In [ ]:
print('TOP 5 CHAMPIONNATS PAR VALEUR MARCHANDE')
print('=' * 60)

df_league = pd.read_sql("""
    SELECT competition_name,
           ROUND(avg_market_value) AS valeur_moyenne,
           player_count
    FROM agg_value_by_league
    ORDER BY avg_market_value DESC
    LIMIT 5
""", conn)

df_league['valeur_moyenne'] = df_league['valeur_moyenne'].apply(
    lambda x: f"{x:,.0f} EUR"
)
print(df_league.to_string(index=False))
print()
print("Insight : La Premier League domine le marche !")

In [ ]:
print('CORRELATION NOTE FIFA vs VALEUR MARCHANDE')
print('=' * 60)

df_fifa = pd.read_sql("""
    SELECT
        CASE
            WHEN overall_rating >= 85 THEN '1. Elite 85+'
            WHEN overall_rating >= 75 THEN '2. Bon 75-84'
            WHEN overall_rating >= 65 THEN '3. Moyen 65-74'
            ELSE '4. Faible moins de 65'
        END AS niveau_fifa,
        ROUND(AVG(latest_market_value)) AS valeur_moyenne,
        COUNT(*) AS nb_joueurs
    FROM fact_player_value
    WHERE overall_rating IS NOT NULL
    GROUP BY niveau_fifa
    ORDER BY niveau_fifa
""", conn)

df_fifa['valeur_moyenne'] = df_fifa['valeur_moyenne'].apply(
    lambda x: f"{x:,.0f} EUR"
)
print(df_fifa.to_string(index=False))
print()
print("Insight : Correlation confirmee entre note FIFA et valeur reelle !")

---
## ÉTAPE 7 — Vérification Sécurité

In [ ]:
print('VERIFICATION DES ROLES POSTGRESQL')
print('=' * 50)

cursor.execute("""
    SELECT rolname, rolcanlogin
    FROM pg_roles
    WHERE rolname IN ('role_spark_etl', 'role_metabase_read')
""")

roles = cursor.fetchall()
for role in roles:
    print(f'  Role : {role[0]} | Peut se connecter : {role[1]}')

print()
print('Test role_metabase_read (lecture seule) :')

conn_read = psycopg2.connect(
    host='localhost', port=5432,
    database='football',
    user='role_metabase_read',
    password='metabase_read_2024'
)
cursor_read = conn_read.cursor()

cursor_read.execute('SELECT COUNT(*) FROM fact_player_value')
count = cursor_read.fetchone()[0]
print(f'  SELECT OK : {count:,} lignes lues')

try:
    cursor_read.execute('DELETE FROM fact_player_value WHERE player_id = 1')
    print('  DELETE : AUTORISE (probleme !)')
except psycopg2.errors.InsufficientPrivilege:
    print('  DELETE REFUSE : role lecture seule fonctionne !')
finally:
    conn_read.rollback()
    conn_read.close()

---
## RÉSUMÉ FINAL

In [ ]:
print('=' * 60)
print('RESUME COMPLET DU PIPELINE')
print('=' * 60)
print()
print('INFRASTRUCTURE')
print('  Docker (8 services)            OK')
print('  Reseau football_net            OK')
print()
print('DONNEES')
print('  Bronze (MinIO CSV)             OK')
print('  Silver (MinIO Parquet)         OK')
print('  Partitionnement par annee      OK')
print('  Gold (PostgreSQL 9 tables)     OK')
print()
print('SECURITE')
print('  role_spark_etl                 OK')
print('  role_metabase_read             OK')
print('  Moindre privilege verifie      OK')
print()
print('RESULTATS METIER')
print('  Pic valeur marchande : 24-26 ans')
print("  Poste le plus valorise : Attack")
print('  Championnat dominant : Premier League')
print('  Correlation FIFA / Transfermarkt : confirmee')
print()
print('INTERFACES')
print('  Airflow  : http://localhost:8081')
print('  MinIO    : http://localhost:9003')
print('  Spark UI : http://localhost:8080')
print('  Metabase : http://localhost:3000')
print()
print('=' * 60)
print('PIPELINE COMPLET ET FONCTIONNEL !')
print('=' * 60)

conn.close()